# 8. Preprocesamiento y Pipeline

**Qué hacemos:** decidir, **solo con el entrenamiento**, cómo limpiar, transformar y codificar las variables, y empaquetar todo en un
`Pipeline` de scikit-learn que se ajusta dentro de cada pliegue de la validación cruzada.

**Por qué:** si la imputación, el escalado o la codificación se calculan con todos los datos, la información de los pliegues de validación (o de la
prueba) se filtra al entrenamiento. Con el `Pipeline` cada pliegue aprende sus propios parámetros.

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

# El OneHotEncoder usa handle_unknown="infrequent_if_exist": una categoría que aparece en validación/prueba y no en el ajuste
# se codifica como ceros (comportamiento deseado y documentado). Se silencia solo ese aviso, también en los procesos de joblib.
import os
_msg_onehot = "Found unknown categories"
warnings.filterwarnings("ignore", message=_msg_onehot, category=UserWarning)
os.environ["PYTHONWARNINGS"] = ",".join(filter(None, [os.environ.get("PYTHONWARNINGS", ""), f"ignore:{_msg_onehot}:UserWarning"]))

pop = pd.read_csv(TABLES / "decision_poblacion.csv").iloc[0]
excluir = bool(pop["excluir_terminales"])
train = load_train(include_terminal=not excluir)
frame = prepare_frame(train)
y, groups = frame[TARGET], frame[GROUP_COL]
cols = choose_columns(frame)

## Población de modelado

La decisión del capítulo 7 se aplica aquí: los encuentros con alta por fallecimiento u hospicio quedan fuera si su tasa de reingreso es
prácticamente cero.

In [2]:
md_out(f"- **Altas terminales excluidas:** {'sí' if excluir else 'no'} "
       f"({es(int(pop['n_excluidos_train']), 0) if excluir else '0'} encuentros de entrenamiento)\n"
       f"- **Encuentros de modelado (entrenamiento):** {es(len(train), 0)}; pacientes: {es(groups.nunique(), 0)}\n"
       f"- **Tasa de reingreso en la población de modelado:** {es(y.mean(), 1, pct=True)}")

- **Altas terminales excluidas:** sí (1.940 encuentros de entrenamiento)
- **Encuentros de modelado (entrenamiento):** 79.473; pacientes: 55.990
- **Tasa de reingreso en la población de modelado:** 11,3 %

## Decisiones de limpieza

Las decisiones se fijan usando únicamente training:

1. Excluir `encounter_id`, `patient_nbr`, `readmitted` y `reingreso_30d` de X.
2. Excluir constantes.
3. Excluir columnas con más de 70 % de faltantes. Este es un **umbral definido
   por el proyecto**, no una regla del profesor, y evita modelar variables
   observadas en menos de 30 % de los encuentros.
4. Derivar `age_mid` y grupos CIE-9 mediante reglas fijas.
5. Imputar, agrupar categorías raras, codificar y escalar dentro del Pipeline.
6. Usar `OneHotEncoder(drop='first')`, de modo que cada coeficiente categórico
   tenga una categoría de referencia explícita.

In [3]:
quitadas = pd.DataFrame([{"variable": k, "motivo": v} for k, v in cols["dropped"].items()])
display(quitadas if len(quitadas) else pd.DataFrame({"resultado": ["ninguna columna eliminada"]}))
md_out(f"Se eliminan **{len(cols['dropped'])}** columnas por ser constantes o tener demasiados faltantes. "
       f"Quedan **{len(cols['num_lin']) + len(cols['num_log'])}** numéricas y **{len(cols['cat'])}** categóricas.")

,variable,motivo
0,weight,faltantes > 70 %
1,examide,constante
2,citoglipton,constante


Se eliminan **3** columnas por ser constantes o tener demasiados faltantes. Quedan **9** numéricas y **35** categóricas.

## Transformación de los conteos con cola larga: ¿`log1p` sí o no?

Las tres variables de visitas previas son muy asimétricas. Se compara, con **validación cruzada agrupada por paciente dentro del entrenamiento**,
el AUC del modelo con y sin `log1p` en esas variables.

**Criterio fijado antes de ver el resultado:** se adopta `log1p` si mejora el AUC medio en al menos **0,001** y en **4 de los 5 pliegues**.

In [4]:
from sklearn.model_selection import cross_validate
folds = cv_folds(y, groups)
todas = cols["num_lin"] + cols["num_log"] + cols["cat"]
X = frame[todas]

def auc_folds(num_lin, num_log):
    pipe = make_model_pipeline(num_lin, num_log, cols["cat"])
    return cross_validate(pipe, X, y, cv=folds, scoring="roc_auc", n_jobs=2, error_score="raise")["test_score"]

sin_log = auc_folds(cols["num_lin"] + cols["num_log"], [])
con_log = auc_folds(cols["num_lin"], cols["num_log"])
dif = con_log - sin_log
usar_log = bool(dif.mean() >= 0.001 and (dif > 0).sum() >= 4)
comp = pd.DataFrame({"pliegue": range(1, len(folds) + 1), "AUC sin log1p": sin_log, "AUC con log1p": con_log, "diferencia": dif})
display(comp.round(4))
md_out(f"AUC medio **sin** `log1p`: {es(sin_log.mean(), 4)}; **con** `log1p`: {es(con_log.mean(), 4)}; diferencia media {es(dif.mean(), 4)}, a favor en "
       f"**{int((dif > 0).sum())} de {len(dif)}** pliegues. **Decisión:** " +
       ("se **usa `log1p`** en las variables de conteo con cola larga." if usar_log else "**no se usa `log1p`**: la mejora no cumple el criterio y se mantiene la escala original."))
pd.DataFrame([{"usar_log1p": usar_log, "auc_sin_log": sin_log.mean(), "auc_con_log": con_log.mean(), "diferencia_media": dif.mean(),
               "folds_a_favor": int((dif > 0).sum()), "criterio": "dif media >= 0,001 y >= 4 de 5 pliegues"}]).to_csv(TABLES / "decision_transformacion.csv", index=False)

final_cols = {"num_lin": cols["num_lin"] + ([] if usar_log else cols["num_log"]), "num_log": cols["num_log"] if usar_log else [], "cat": cols["cat"]}
pd.DataFrame([{"variable": v, "grupo": g} for g, vs in final_cols.items() for v in vs]).to_csv(TABLES / "columnas_modelo.csv", index=False)

,pliegue,AUC sin log1p,AUC con log1p,diferencia
0,1,0.6596,0.6581,-0.0015
1,2,0.6560,0.6581,0.0021
2,3,0.6605,0.6648,0.0043
3,4,0.6699,0.6720,0.0021
4,5,0.6571,0.6599,0.0028


AUC medio **sin** `log1p`: 0,6606; **con** `log1p`: 0,6626; diferencia media 0,0020, a favor en **4 de 5** pliegues. **Decisión:** se **usa `log1p`** en las variables de conteo con cola larga.

## Pipeline final

El `ColumnTransformer` tiene ramas numéricas (mediana, transformación cuando
proceda y escalado) y categóricas (`Missing`, agrupación de niveles infrecuentes
y OneHotEncoder con primera categoría como referencia). Todos los parámetros se
aprenden dentro de cada fold. Después se ajusta la regresión logística base.

In [5]:
pipe = make_model_pipeline(final_cols["num_lin"], final_cols["num_log"], final_cols["cat"])
pipe.fit(X, y)
nombres = pipe.named_steps["prep"].get_feature_names_out()
por_rama = pd.Series([n.split("__")[0] for n in nombres]).value_counts()
md_out(f"Tras el ajuste en el entrenamiento, el Pipeline produce **{len(nombres)} columnas**: "
       + ", ".join(f"{k} = {v}" for k, v in por_rama.items()) + ".\n\n"
       "Ejemplos de columnas creadas: " + ", ".join(f"`{n}`" for n in list(nombres[:3]) + [n for n in nombres if 'infrequent' in n][:2]) + ".")
referencias = onehot_reference_table(pipe, final_cols["cat"])
referencias.to_csv(TABLES / "categorias_referencia_onehot.csv", index=False)
display(referencias.head(12))
pipe

Tras el ajuste en el entrenamiento, el Pipeline produce **120 columnas**: cat = 111, num = 6, numlog = 3.

Ejemplos de columnas creadas: `num__time_in_hospital`, `num__num_lab_procedures`, `num__num_procedures`, `cat__race_infrequent_sklearn`, `cat__gender_infrequent_sklearn`.

,variable,categoría_referencia
0,race,AfricanAmerican
1,gender,Female
2,admission_type_id,1
3,discharge_disposition_id,1
4,admission_source_id,1
5,payer_code,BC
6,medical_specialty,Cardiology
7,max_glu_serum,>200
8,A1Cresult,>7
9,metformin,No


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('sc',
                                                                   StandardScaler())]),
                                                  ['time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_diagnoses',
                                                   'age_mid']),
                                                 ('numlog',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log',
                                                                   FunctionTransformer(fea...
                                                   'chlorpropamide',
                                                   'glimepiride',
                                                   'acetohexamide', 'glipizide',
                                                   'glyburide', 'tolbutamide',
                                                   'pioglitazone',
                                                   'rosiglitazone', 'acarbose',
                                                   'miglitol', 'troglitazone',
                                                   'tolazamide', 'insulin',
                                                   'glyburide-metformin',
                                                   'glipizide-metformin',
                                                   'glimepiride-pioglitazone',
                                                   'metformin-rosiglitazone',
                                                   'metformin-pioglitazone', ...])])),
                ('clf', LogisticRegression(max_iter=2000, random_state=42))])

In [6]:
# Comprobación: el Pipeline aprende parámetros distintos en cada pliegue (no se comparten entre entrenamiento y validación)
tr, va = folds[0]
p_a, p_b = make_model_pipeline(final_cols["num_lin"], final_cols["num_log"], final_cols["cat"]), make_model_pipeline(final_cols["num_lin"], final_cols["num_log"], final_cols["cat"])
p_a.fit(X.iloc[tr], y.iloc[tr]); p_b.fit(X, y)
media_a = p_a.named_steps["prep"].named_transformers_["num"].named_steps["sc"].mean_[:3]
media_b = p_b.named_steps["prep"].named_transformers_["num"].named_steps["sc"].mean_[:3]
md_out("**Verificación anti-fuga.** Las medias usadas para estandarizar las primeras tres variables son " + ", ".join(es(v, 3) for v in media_a) +
       " si el Pipeline se ajusta solo con el pliegue de entrenamiento y " + ", ".join(es(v, 3) for v in media_b) +
       " si se ajustara con todos los datos: son distintas, es decir, **cada pliegue aprende sus propios parámetros**.")

**Verificación anti-fuga.** Las medias usadas para estandarizar las primeras tres variables son 4,381, 42,955, 1,339 si el Pipeline se ajusta solo con el pliegue de entrenamiento y 4,381, 42,892, 1,335 si se ajustara con todos los datos: son distintas, es decir, **cada pliegue aprende sus propios parámetros**.

## Resumen de decisiones de preprocesamiento

| Paso | Decisión | Dónde se calcula |
|---|---|---|
| Población | Excluir altas por fallecimiento u hospicio (regla clínica) | Capítulo 7 |
| Identificadores y objetivo original | Excluir | Capítulo 7 |
| Constantes y > 70 % de faltantes | Excluir (umbral metodológico del proyecto) | Entrenamiento |
| Diagnósticos | Agrupar en capítulos CIE-9 | Regla fija |
| Edad | Punto medio del rango | Regla fija |
| Faltantes categóricos | Categoría `Missing` | Pipeline |
| Categorías raras (< 1 %) | Agrupar; one-hot con primera categoría de referencia | Pipeline (entrenamiento) |
| Conteos con cola larga | `log1p` según el criterio de arriba | Validación cruzada agrupada |
| Escalado | Estandarización | Pipeline (entrenamiento) |